# 05 - Synthetic GPR data generation (controlled)

This notebook builds the controlled synthetic dataset for the transfer study (H1). Run it on the
RTX 5090 machine, top to bottom, with the `paleo-gpr-ml` kernel.

**What it does, in order**
1. Check the machine sees the GPU and gprMax.
2. Generate the controlled gprMax models (void, bone, anti_bone, null).
3. Run gprMax on them and merge each B-scan.
4. The fidelity gate: confirm the synthetic physics is right before I trust the data.
5. Convert the B-scans to images plus YOLO labels.
6. Look at the result.

**The one rule:** do not move past step 4 until the fidelity gate passes. The whole hypothesis
rests on the synthetic physics being correct, so this is the checkpoint I do not skip. The
physics I am checking against is in `docs/notes/08_forward_model_1d_validation.md`, and the plan
this feeds is `docs/experiments/experiment_03_transfer.md`.

**Time:** generating the `.in` files is instant. The gprMax runs are the slow part. With the GPU
each 2D B-scan is seconds to a couple of minutes depending on the model. The 32-model pilot
should finish in well under an hour. The full grid scales from there.

**Outputs:** everything lands under `data/processed/synthetic/` (gitignored, regenerable). The
end state is one image and one YOLO label per scene, organized by target type.

## 0. Check the machine

I want to see the RTX 5090 with Blackwell compute capability `(12, 0)` and a nightly torch, plus
gprMax importable. If torch does not see the GPU, the wrong wheel is installed. See
`docs/notes/12_cuda_runbook.md` for the fix.

**Expected output:** a `2.x.dev` torch, `NVIDIA GeForce RTX 5090`, capability `(12, 0)`, and a
gprMax version. Anything else means stop and fix the environment first.

In [ ]:
import platform
import torch

print("python:", platform.python_version())
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    cap = torch.cuda.get_device_capability(0)
    print("compute capability:", cap)
    if cap[0] < 12:
        print("WARNING: expected Blackwell (12, 0) for the RTX 5090. Wrong torch wheel?")
else:
    print("WARNING: CUDA not available. On the 5090 this is the wrong torch wheel.")
    print("See docs/notes/12_cuda_runbook.md, install torch from the nightly cu128 index.")

try:
    import gprMax
    print("gprMax:", getattr(gprMax, "__version__", "imported, version unknown"))
except Exception as e:
    print("gprMax not importable:", e)
    print("Build it from source, see docs/notes/12_cuda_runbook.md.")

## 1. Generate the controlled models

The design is the part that makes the result causal. For every scene I write the same 2D model
four times, once per target type, changing only the target dielectric:

- **void** (eps 1): air-filled, like my real cavities. Low permittivity, positive top reflection.
- **bone** (eps ~9): fossil-like. High permittivity, negative top reflection.
- **anti_bone**: a low-permittivity target whose reflection magnitude matches bone but with the
  opposite sign. This is the control that isolates polarity from contrast magnitude.
- **null** (eps = host): no contrast. A negative control that should be nearly invisible.

Everything else (geometry, depth, antenna, frequency) is identical across the four, so any
difference in how a detector treats them comes from the dielectric alone.

**Expected output:** 32 `.in` files plus a `manifest.csv` in
`data/processed/synthetic/controlled/`. The manifest has the ground-truth target position for
each model, which the converter later uses to draw the box.

In [ ]:
from pathlib import Path

import pandas as pd
import yaml

from src.data.generate_gprmax_models import generate

cfg = yaml.safe_load(open("configs/synthetic_controlled.yaml"))
out_dir = Path("data/processed/synthetic/controlled")
manifest_path = generate(cfg, out_dir)

manifest = pd.read_csv(manifest_path)
print(f"{len(manifest)} models written to {out_dir}")
manifest[["scene_id", "target_type", "eps_target", "depth_m", "freq_mhz", "n_traces"]].head(8)

### Sanity check: only the dielectric changes

To prove the control is real, diff the bone model against the anti_bone model for the same scene.
The only line that should differ is the target `#material` permittivity (9 vs 1.778). Same
domain, same cylinder, same antenna, same everything else.

In [ ]:
import difflib

a = (out_dir / "dry_sand_d010_f0400__bone.in").read_text().splitlines()
b = (out_dir / "dry_sand_d010_f0400__anti_bone.in").read_text().splitlines()
diff = [l for l in difflib.unified_diff(a, b, lineterm="")
        if l[:1] in "+-" and not l.startswith(("+++", "---"))]
print("\n".join(diff))
print("\nExpected: only the target #material permittivity differs.")

## 2. Run gprMax and merge each B-scan

gprMax runs a finite-difference time-domain simulation of the wave. For a B-scan I run the model
with `-n <n_traces>`, which steps the antenna across the line and writes one output file per
trace. Then I merge those into a single B-scan file.

`-gpu` uses the CUDA solver, which is the reason to be on this machine. If pycuda or the CUDA
toolkit gives trouble on Blackwell, drop `-gpu` and the CPU solver produces the same B-scan, just
slower.

**Note:** the merge tool module path can vary by gprMax version. If `tools.outputfiles_merge`
is not found, check the gprMax docs for the current path. It writes `{stem}_merged.out`.

**Expected per model:** `{stem}1.out ... {stem}N.out` appear, then `{stem}_merged.out` after the
merge. The merged file is the B-scan I work with.

In [ ]:
import subprocess
import sys


def run_gprmax(in_file: Path, n_traces: int, use_gpu: bool = True):
    cmd = [sys.executable, "-m", "gprMax", str(in_file), "-n", str(n_traces)]
    if use_gpu:
        cmd += ["-gpu"]
    print("running:", " ".join(cmd))
    subprocess.run(cmd, check=True)


def merge_bscan(in_file: Path, remove_parts: bool = True):
    base = str(in_file.with_suffix(""))  # gprMax wants the basename without extension
    cmd = [sys.executable, "-m", "tools.outputfiles_merge", base]
    if remove_parts:
        cmd += ["--remove-files"]
    subprocess.run(cmd, check=True)


def merged_path(in_file: Path) -> Path:
    return in_file.with_name(in_file.stem + "_merged.out")

### Smoke test on one model first

Before running all 32, run one and confirm the merged file appears. This catches environment
problems early instead of after a long batch.

In [ ]:
row0 = manifest.iloc[0]
in0 = out_dir / row0["in_file"]
run_gprmax(in0, int(row0["n_traces"]), use_gpu=True)
merge_bscan(in0)
print("merged file exists:", merged_path(in0).exists())

### Run the rest

This is the slow cell. It skips any model already merged, so it is safe to re-run if it stops.
Watch the progress bar. Each model is independent.

In [ ]:
from tqdm.notebook import tqdm

for _, row in tqdm(manifest.iterrows(), total=len(manifest)):
    in_f = out_dir / row["in_file"]
    if merged_path(in_f).exists():
        continue
    run_gprmax(in_f, int(row["n_traces"]), use_gpu=True)
    merge_bscan(in_f)
print("done")

## 3. Fidelity gate (the checkpoint)

This is the step that decides whether the synthetic data is trustworthy. I check two things
against the 1D physics from note 08:

1. **Polarity.** Bone (high permittivity) should reflect **negative** at the top of the target.
   Void (low permittivity) should reflect **positive**. This is the whole basis of H1.
2. **Shape.** A buried point-like target should produce a **hyperbola** in the B-scan, apex over
   the target.

The automated polarity read looks in a window around the expected target time (computed from
depth), so it skips the direct wave at the very top. Still, look at the images yourself and
confirm you see a clean hyperbola with the right apex polarity.

**Expected:** bone top polarity -1, void top polarity +1, and a visible hyperbola in each image.

**If this fails:** stop. Do not convert or train. Check the dielectric values, the time window,
and the geometry in `configs/synthetic_controlled.yaml`, then re-run.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.data.process_gprmax_output import _host_velocity_m_per_ns, read_merged_bscan, time_gain


def load(stem: str):
    return read_merged_bscan(merged_path(out_dir / f"{stem}.in"))


def target_top_polarity(bscan, dt_s, depth_m, host_eps):
    v = _host_velocity_m_per_ns(host_eps)            # m/ns
    apex = int((2 * depth_m / v) / (dt_s * 1e9))     # twt at the apex, in samples
    col = bscan.shape[1] // 2                         # target is centered
    lo, hi = max(0, apex - 30), apex + 30
    w = bscan[lo:hi, col]
    k = int(np.argmax(np.abs(w)))
    return float(np.sign(w[k]))


bone, dt = load("dry_sand_d010_f0400__bone")
void, _ = load("dry_sand_d010_f0400__void")

pb = target_top_polarity(bone, dt, 0.10, 4.0)
pv = target_top_polarity(void, dt, 0.10, 4.0)
print(f"bone top polarity: {pb:+.0f}  (expect -1)")
print(f"void top polarity: {pv:+.0f}  (expect +1)")
gate = (pb < 0) and (pv > 0)
print("FIDELITY GATE:", "PASS" if gate else "FAIL -- do not process or train until this passes")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (name, bs) in zip(axes, [("bone", bone), ("void", void)]):
    ax.imshow(time_gain(bs), aspect="auto", cmap="gray")
    ax.set_title(name)
    ax.set_xlabel("trace (antenna position)")
    ax.set_ylabel("time sample (depth)")
plt.tight_layout()
plt.show()

## 4. Convert to images plus YOLO labels

Now turn each merged B-scan into a grayscale image and a YOLO label box. The box comes from the
known target geometry in the manifest, not from guessing, because I placed the target myself.
Null scenes get an empty label (a background image with no object), which is the YOLO convention.

**Expected output:** under `data/processed/synthetic/images/<target_type>/images` and `/labels`,
one PNG and one `.txt` per scene, plus a `processed_manifest.csv`.

In [ ]:
from src.data.process_gprmax_output import process_manifest

img_root = Path("data/processed/synthetic/images")
processed_manifest = process_manifest(manifest_path, out_dir, img_root, image_size=416)

proc = pd.read_csv(processed_manifest)
print(proc.groupby("target_type")["has_target"].agg(["count", "sum"]).rename(
    columns={"count": "scenes", "sum": "with_box"}))

### Look at the dataset

A few images per target type with the YOLO box drawn on. Confirm the box sits over the hyperbola
apex and limbs, and that null images have no box. Bone and void should look like clean hyperbolas
with opposite apex polarity.

In [ ]:
import matplotlib.patches as patches
from PIL import Image


def show(ttype, k=3):
    d = img_root / ttype
    imgs = sorted((d / "images").glob("*.png"))[:k]
    if not imgs:
        print(f"no images for {ttype}")
        return
    fig, axes = plt.subplots(1, len(imgs), figsize=(4 * len(imgs), 4))
    if len(imgs) == 1:
        axes = [axes]
    for ax, ip in zip(axes, imgs):
        im = Image.open(ip)
        W, H = im.size
        ax.imshow(im, cmap="gray")
        ax.set_title(f"{ttype}: {ip.stem}", fontsize=8)
        ax.axis("off")
        txt = (d / "labels" / f"{ip.stem}.txt").read_text().strip()
        if txt:
            _, cx, cy, w, h = map(float, txt.split())
            ax.add_patch(patches.Rectangle(
                ((cx - w / 2) * W, (cy - h / 2) * H), w * W, h * H,
                edgecolor="red", facecolor="none", lw=2))
    plt.tight_layout()
    plt.show()


for t in ["bone", "void", "anti_bone", "null"]:
    show(t)

## 5. What is ready, and what is next

If the fidelity gate passed and the images look right, I now have the controlled synthetic sets
the transfer study needs: matched void, bone, anti_bone, and null scenes that differ only in the
target dielectric.

**This feeds `docs/experiments/experiment_03_transfer.md`:**
- C0a / C0b: the same-domain ceilings (train and test on void, or on bone).
- C1: naive transfer, train on void, test on bone (H1a, does the gap exist).
- C2: the polarity control, bone vs anti_bone (H1b, is polarity the cause).
- Cbase: the polarity matched filter (`src/models/run_polarity_baseline.py`) for comparison.

**Next notebook:** train the detectors for C0 to C2 and read the results against the
pre-registered decision rules. Before reporting anything, run the `research-integrity-check`.

Scale-up later: increase the grid in `configs/synthetic_controlled.yaml` (more depths, hosts,
positions, seeds) and add heterogeneous soil with fixed per-scene seeds so the clutter is
identical across target types.